In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use("TkAgg")

from preprocesare import redimensionare, Imagini, centrare_date
from matematica import svd


def calculeaza_sosia_dupa_selectie(imagine_de_la_interfata, k_componente):
    if imagine_de_la_interfata is None:
        return None

    img = Image.fromarray(imagine_de_la_interfata).convert("L")
    IMG = redimensionare(np.array(img), linii, coloane)

    v_tu = IMG.flatten().astype(np.float64)
    if np.linalg.norm(v_tu) == 0:
        return None
    v_tu_normat = v_tu / np.linalg.norm(v_tu)
    v_tu_centrat = v_tu_normat - Fata_medie

    k = int(k_componente)
    U_redus = U[:, :k]
    W_redus = W[:k, :]

    w_tu = U_redus.T @ v_tu_centrat
    distante = np.linalg.norm(W_redus - w_tu[:, np.newaxis], axis=0)
    index_minim = np.argmin(distante)

    sosia_vector = X[:, index_minim]
    sosia_matrice = sosia_vector.reshape(linii, coloane)
    sosia_0_255 = (
        (sosia_matrice - sosia_matrice.min())
        / (sosia_matrice.max() - sosia_matrice.min())
        * 255
    ).astype(np.uint8)

    return sosia_0_255


Fete, n, linii, coloane, target, target_names = Imagini()

MAX_PER_PERSOANA = 5
selectati = []
contor = {}
for i in range(n):
    persoana = target[i]
    if persoana not in contor:
        contor[persoana] = 0
    if contor[persoana] < MAX_PER_PERSOANA:
        selectati.append(i)
        contor[persoana] += 1
    if len(selectati) >= 250:
        break

ok = 0
X_list = []
target_selectati = []
for i in selectati:
    if linii > 200 or coloane > 200:
        img_prelucrata = redimensionare(Fete[i], 200, 200)
        ok = 1
    else:
        img_prelucrata = Fete[i]
    v_brut = img_prelucrata.flatten().astype(np.float64)
    if np.linalg.norm(v_brut) == 0:
        continue
    v_normat = v_brut / np.linalg.norm(v_brut)
    X_list.append(v_normat)
    target_selectati.append(target[i])

X = np.array(X_list).T
if ok == 1:
    linii, coloane = 200, 200
A, Fata_medie = centrare_date(X)

"""
plt.imshow(Fata_medie.reshape(linii, coloane), cmap='gray')
plt.show()
"""

U, S, Vt = svd(A)

"""
Afișăm prima Eigenface (cea mai importantă)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(U[:, 0].reshape(linii, coloane), cmap='gray')
plt.title("Eigenface 1 (Cea mai mare valoare proprie)")

Afișăm a doua Eigenface (următoarea ca importanță)
plt.subplot(1, 2, 2)
plt.imshow(U[:, 1].reshape(linii, coloane), cmap='gray')
plt.title("Eigenface 2")
plt.show()
"""

W = U.T @ A
"""
# 1. Extragem ponderile primei fețe (prima coloană din W)
w1 = W[:, 0] 
# 2. Reconstruim fața în spațiul pixelilor
# Înmulțim matricea U (Eigenfaces) cu vectorul de ponderi w1
fata_reconstruita_centrata =  U[:, :100] @ w1
# 3. Adăugăm înapoi Fața Medie (psi) pentru a reveni la aspectul original
fata_finala = fata_reconstruita_centrata + Fata_medie.flatten()
# 4. Afișăm rezultatul
plt.imshow(fata_finala.reshape(linii, coloane), cmap='gray')
plt.title("Prima față reconstruită din ponderile W")
plt.show() 
"""

demo = gr.Interface(
    fn=calculeaza_sosia_dupa_selectie,
    inputs=[
        gr.Image(label="1. Alege/Trage poza ta aici"),
        gr.Slider(
            minimum=1,
            maximum=150,
            value=15,
            step=1,
            label="Număr Componente Principale (k)",
        ),
    ],
    outputs=gr.Image(label="2. Sosia ta calculată", image_mode="L"),
    title="Recunoaștere Facială",
    description="Reglează parametrul k pentru a modifica numărul de caracteristici vectoriale folosite la comparare.",
)

demo.launch()